In [0]:
%run "../../utils"

In [0]:
df_state = (
    spark.table(f"{silver_schema}.alt_fuel_stations")
      .groupBy("state")
      .agg(spark_count("*").alias("station_count"))
      .orderBy(col("station_count").desc())
)

df_state.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.stations_by_state")

In [0]:
df_fuel = (
    spark.table(f"{silver_schema}.alt_fuel_stations")
      .groupBy("fuel_type_code")
      .agg(spark_count("*").alias("station_count"))
      .orderBy(col("station_count").desc())
)

df_fuel.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.stations_by_fuel_type")

In [0]:
from pyspark.sql.functions import year

df_year = (
    spark.table(f"{silver_schema}.alt_fuel_stations")
      .filter(col("open_date").isNotNull())
      .withColumn("open_year", year(col("open_date")))
      .groupBy("open_year")
      .agg(spark_count("*").alias("stations_opened"))
      .orderBy(col("open_year").asc())
)

df_year.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}stations_opened_by_year")